# PRAVIT Video-Pipeline

Gehärtete Fassung. Sie ist so gebaut, dass **ein Fehler nie den ganzen Lauf
mitreißt** und **keine bezahlten Ergebnisse verlorengehen**.

Vier Dinge, die die vorige Fassung nicht hatte:

1. **Jede Szene wird einzeln abgesichert.** Fällt Szene 3 aus, laufen 4 bis 8
   trotzdem weiter.
2. **Der Fortschritt liegt auf Google Drive**, nicht im Arbeitsspeicher. Die
   Sitzung darf abstürzen, der Stand bleibt.
3. **Jede `task_id` wird sofort protokolliert** — bevor auf das Ergebnis
   gewartet wird. Genau daran ist der letzte Lauf gescheitert: Die Aufträge
   waren bezahlt, aber niemand wusste mehr, wie sie hießen.
4. **Kostenvorschau vor dem Start.** Es läuft nichts los, bevor du nicht
   siehst, was es kostet, und ausdrücklich bestätigst.

> **Zuerst Abschnitt 6 ausführen**, wenn du Ergebnisse aus dem abgestürzten
> Lauf zurückholen willst. Das kostet keine neuen Credits.

## 1 · Einrichtung

Der Zugang steht in den Colab-Secrets (Schlüsselsymbol links), nicht im
Notebook. Sonst liegt er in jeder geteilten Kopie offen.

In [ ]:
# Falls noch nicht vorhanden
!pip install -q requests

import json
import os
import time
from dataclasses import dataclass, asdict, field
from datetime import datetime
from pathlib import Path

import requests

# Google Drive einhängen – dort liegt der Fortschritt und überlebt jeden Absturz.
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    ARBEIT = Path('/content/drive/MyDrive/pravit-video')
    API_SCHLUESSEL = userdata.get('VIDEO_API_KEY')
except Exception as fehler:
    # Damit sich das Notebook auch lokal ausprobieren lässt.
    print(f'Kein Colab erkannt ({fehler}) – arbeite lokal.')
    ARBEIT = Path('./pravit-video')
    API_SCHLUESSEL = os.environ.get('VIDEO_API_KEY', '')

ARBEIT.mkdir(parents=True, exist_ok=True)
(ARBEIT / 'videos').mkdir(exist_ok=True)

ZUSTAND_DATEI = ARBEIT / 'zustand.json'
PROTOKOLL = ARBEIT / 'auftraege.jsonl'

print(f'Arbeitsordner: {ARBEIT}')
print(f'Schlüssel gesetzt: {bool(API_SCHLUESSEL)}')

## 2 · Anbieter

Alles Anbieterspezifische steckt in dieser einen Klasse. Wechselst du den
Dienst, änderst du hier drei Methoden — der Rest des Notebooks bleibt.

**Anzupassen:** `BASIS`, die Pfade und die Feldnamen in `auftrag_starten`
und `ergebnis_holen`. Die Namen unten sind die verbreiteten; prüf sie einmal
gegen die Dokumentation deines Anbieters.

In [ ]:
BASIS = 'https://api.example-video.com/v1'   # ← anpassen
MODELL = 'video-01'                          # ← anpassen
PREIS_JE_SZENE = 0.35                        # ← in Euro, für die Vorschau


class Anbieter:
    """Dünne Hülle um die Video-API."""

    def __init__(self, schluessel: str):
        self.kopf = {
            'Authorization': f'Bearer {schluessel}',
            'Content-Type': 'application/json',
        }

    def auftrag_starten(self, prompt: str, sekunden: int) -> str:
        """Startet die Erzeugung und gibt die Auftragskennung zurück."""
        antwort = requests.post(
            f'{BASIS}/video/generation',
            headers=self.kopf,
            json={
                'model': MODELL,
                'prompt': prompt,
                'duration': sekunden,
                'aspect_ratio': '9:16',
            },
            timeout=60,
        )
        antwort.raise_for_status()
        daten = antwort.json()
        kennung = daten.get('task_id') or daten.get('id')
        if not kennung:
            raise RuntimeError(f'Keine Auftragskennung in der Antwort: {daten}')
        return kennung

    def ergebnis_holen(self, kennung: str) -> dict:
        """Fragt den Stand ab. Gibt {status, url} zurück."""
        antwort = requests.get(
            f'{BASIS}/video/generation/{kennung}',
            headers=self.kopf,
            timeout=60,
        )
        antwort.raise_for_status()
        daten = antwort.json()
        return {
            'status': daten.get('status', 'unbekannt'),
            'url': daten.get('video_url') or daten.get('url'),
            'roh': daten,
        }


anbieter = Anbieter(API_SCHLUESSEL)
print('Anbieter bereit.')

## 3 · Zustand

Zwei Dateien auf Drive:

* `zustand.json` — je Szene: Kennung, Status, Adresse des Ergebnisses
* `auftraege.jsonl` — jede vergebene Auftragskennung, **sofort** angehängt

Die zweite ist die Lebensversicherung. Selbst wenn `zustand.json` beschädigt
wird, stehen alle bezahlten Aufträge noch in der Protokolldatei.

In [ ]:
def zustand_laden() -> dict:
    if ZUSTAND_DATEI.exists():
        try:
            return json.loads(ZUSTAND_DATEI.read_text())
        except json.JSONDecodeError:
            # Beschädigt – beiseitelegen statt überschreiben.
            beiseite = ZUSTAND_DATEI.with_suffix(f'.kaputt-{int(time.time())}.json')
            ZUSTAND_DATEI.rename(beiseite)
            print(f'zustand.json war beschädigt, liegt jetzt als {beiseite.name}')
    return {}


def zustand_sichern(zustand: dict) -> None:
    # Erst daneben schreiben, dann umbenennen: Bricht Colab mitten im
    # Schreiben ab, ist die alte Datei noch heil.
    vorlaeufig = ZUSTAND_DATEI.with_suffix('.tmp')
    vorlaeufig.write_text(json.dumps(zustand, indent=2, ensure_ascii=False))
    vorlaeufig.replace(ZUSTAND_DATEI)


def auftrag_protokollieren(szene_id: str, kennung: str, prompt: str) -> None:
    """Hängt die Kennung sofort an – vor jedem Warten."""
    eintrag = {
        'zeit': datetime.now().isoformat(timespec='seconds'),
        'szene': szene_id,
        'task_id': kennung,
        'prompt': prompt[:200],
    }
    with PROTOKOLL.open('a', encoding='utf-8') as datei:
        datei.write(json.dumps(eintrag, ensure_ascii=False) + '\n')
        datei.flush()
        os.fsync(datei.fileno())   # Wirklich auf die Platte, nicht nur in den Puffer.


zustand = zustand_laden()
print(f'{len(zustand)} Szenen im Zustand.')

## 4 · Szenen

Hier stehen die Aufnahmen. Die Texte stammen aus den Storyboards
(`docs/D-Storyboards.md`) — anpassen, ergänzen, weglassen.

In [ ]:
SZENEN = [
    {
        'id': 'erklaer_01',
        'sekunden': 6,
        'prompt': (
            'Nahaufnahme einer Hantelstange auf einer Ablage in einem modernen, '
            'dunklen Fitnessstudio. Weiches Seitenlicht, Staub in der Luft, '
            'flache Schärfentiefe, langsame Kamerafahrt nach rechts. '
            'Kinowirkung, ruhig, kein Text.'
        ),
    },
    {
        'id': 'erklaer_02',
        'sekunden': 6,
        'prompt': (
            'Junger Trainer schaut auf ein Smartphone, konzentriert, nickt kurz. '
            'Dunkles Studio im Hintergrund, unscharf. Natürliches Licht von '
            'links, Halbnahaufnahme, ruhige Handkamera.'
        ),
    },
    {
        'id': 'werbung_01',
        'sekunden': 5,
        'prompt': (
            'Schwarzer Hintergrund, eine einzelne weiße Hantelstange schwebt '
            'langsam ins Bild und kommt zur Ruhe. Minimalistisch, hoher '
            'Kontrast, weiches Licht von oben. Kein Text.'
        ),
    },
]

print(f'{len(SZENEN)} Szenen vorbereitet.')

## 5 · Kostenvorschau

Zuerst rechnen, dann bestätigen, dann erst starten. Bereits fertige Szenen
werden nicht erneut berechnet.

In [ ]:
offen = [s for s in SZENEN if zustand.get(s['id'], {}).get('status') != 'fertig']
kosten = len(offen) * PREIS_JE_SZENE

print(f'Szenen gesamt:  {len(SZENEN)}')
print(f'Schon fertig:   {len(SZENEN) - len(offen)}')
print(f'Zu erzeugen:    {len(offen)}')
print(f'Kosten etwa:    {kosten:.2f} EUR')
print()
for s in offen:
    print(f"  · {s['id']:14s} {s['sekunden']}s")

BESTAETIGT = False   # ← auf True setzen, wenn die Kosten in Ordnung sind

## 6 · Erzeugen

Jede Szene für sich abgesichert. Fällt eine aus, wird sie vermerkt und die
nächste beginnt — der Lauf bricht nicht ab.

Erneutes Ausführen ist gefahrlos: Fertige Szenen werden übersprungen, und
eine Szene mit bekannter Auftragskennung wird abgefragt statt neu bestellt.
Das ist die Stelle, an der beim letzten Mal die Credits verbrannt sind.

In [ ]:
WARTE_SEKUNDEN = 15
MAX_WARTEN = 40      # 40 x 15s = 10 Minuten je Szene


def szene_erzeugen(szene: dict, zustand: dict) -> None:
    kennung_bekannt = zustand.get(szene['id'], {}).get('task_id')

    if kennung_bekannt:
        print(f"  {szene['id']}: Auftrag {kennung_bekannt} existiert – frage nur ab.")
        kennung = kennung_bekannt
    else:
        kennung = anbieter.auftrag_starten(szene['prompt'], szene['sekunden'])
        # SOFORT protokollieren. Alles danach darf scheitern, ohne dass die
        # bezahlte Kennung verlorengeht.
        auftrag_protokollieren(szene['id'], kennung, szene['prompt'])
        zustand[szene['id']] = {'task_id': kennung, 'status': 'laeuft'}
        zustand_sichern(zustand)
        print(f"  {szene['id']}: gestartet als {kennung}")

    for versuch in range(MAX_WARTEN):
        ergebnis = anbieter.ergebnis_holen(kennung)
        status = str(ergebnis['status']).lower()

        if status in ('success', 'succeeded', 'completed', 'finished') and ergebnis['url']:
            ziel = ARBEIT / 'videos' / f"{szene['id']}.mp4"
            daten = requests.get(ergebnis['url'], timeout=300)
            daten.raise_for_status()
            ziel.write_bytes(daten.content)
            zustand[szene['id']] = {
                'task_id': kennung,
                'status': 'fertig',
                'datei': str(ziel),
                'url': ergebnis['url'],
            }
            zustand_sichern(zustand)
            print(f"  {szene['id']}: fertig → {ziel.name}")
            return

        if status in ('failed', 'error'):
            zustand[szene['id']] = {
                'task_id': kennung,
                'status': 'fehlgeschlagen',
                'grund': str(ergebnis['roh'])[:300],
            }
            zustand_sichern(zustand)
            print(f"  {szene['id']}: fehlgeschlagen")
            return

        time.sleep(WARTE_SEKUNDEN)

    # Zeit abgelaufen – Kennung bleibt gespeichert, Abschnitt 7 holt es später.
    zustand[szene['id']] = {'task_id': kennung, 'status': 'wartet_noch'}
    zustand_sichern(zustand)
    print(f"  {szene['id']}: dauert länger – Kennung gemerkt, später mit Abschnitt 7 holen.")


if not BESTAETIGT:
    print('Nicht bestätigt. Setz BESTAETIGT = True in Abschnitt 5.')
else:
    for szene in SZENEN:
        if zustand.get(szene['id'], {}).get('status') == 'fertig':
            print(f"  {szene['id']}: schon fertig, übersprungen")
            continue
        try:
            szene_erzeugen(szene, zustand)
        except Exception as fehler:
            # Der entscheidende Punkt: Ein Fehler beendet nur diese Szene.
            print(f"  {szene['id']}: Fehler – {fehler}")
            eintrag = zustand.get(szene['id'], {})
            eintrag['status'] = 'fehlgeschlagen'
            eintrag['grund'] = str(fehler)[:300]
            zustand[szene['id']] = eintrag
            zustand_sichern(zustand)

    print()
    print('Durchlauf beendet.')

## 7 · Rettung — Ergebnisse ohne neue Credits holen

Für den abgestürzten Lauf. Trag unten die Auftragskennungen ein, die du noch
hast (aus der alten Ausgabe, dem Anbieter-Dashboard oder `auftraege.jsonl`).

Fertige Aufträge liegen bei den meisten Anbietern **7 bis 30 Tage** bereit.
Was in dieser Zeit abgeholt wird, kostet nichts extra — es ist längst bezahlt.

In [ ]:
# Kennungen aus dem abgestürzten Lauf. Entweder von Hand eintragen …
ALTE_KENNUNGEN = [
    # 'task_abc123',
    # 'task_def456',
]

# … oder alles nehmen, was je protokolliert wurde.
NIMM_AUS_PROTOKOLL = True

kennungen = list(ALTE_KENNUNGEN)
if NIMM_AUS_PROTOKOLL and PROTOKOLL.exists():
    for zeile in PROTOKOLL.read_text(encoding='utf-8').splitlines():
        try:
            kennungen.append(json.loads(zeile)['task_id'])
        except Exception:
            continue

kennungen = list(dict.fromkeys(k for k in kennungen if k))   # doppelte raus, Reihenfolge bleibt
print(f'{len(kennungen)} Kennungen zu prüfen.\n')

gerettet = 0
for kennung in kennungen:
    try:
        ergebnis = anbieter.ergebnis_holen(kennung)
        status = str(ergebnis['status']).lower()

        if status in ('success', 'succeeded', 'completed', 'finished') and ergebnis['url']:
            ziel = ARBEIT / 'videos' / f'gerettet_{kennung}.mp4'
            if ziel.exists():
                print(f'  {kennung}: liegt schon vor')
                continue
            daten = requests.get(ergebnis['url'], timeout=300)
            daten.raise_for_status()
            ziel.write_bytes(daten.content)
            gerettet += 1
            print(f'  {kennung}: gerettet → {ziel.name}')
        else:
            print(f'  {kennung}: Status {status}')
    except Exception as fehler:
        print(f'  {kennung}: nicht abrufbar – {fehler}')

print(f'\n{gerettet} Videos zurückgeholt, ohne neue Credits.')

## 8 · Stand

In [ ]:
zustand = zustand_laden()

if not zustand:
    print('Noch nichts erzeugt.')
else:
    for szene_id, eintrag in sorted(zustand.items()):
        zeichen = {
            'fertig': 'OK  ',
            'laeuft': '... ',
            'wartet_noch': '... ',
            'fehlgeschlagen': 'FEHL',
        }.get(eintrag.get('status'), '?   ')
        print(f"{zeichen} {szene_id:16s} {eintrag.get('status'):16s} {eintrag.get('task_id', '')}")

dateien = sorted((ARBEIT / 'videos').glob('*.mp4'))
print(f'\n{len(dateien)} Videodateien in {ARBEIT / "videos"}')
for d in dateien:
    print(f'  · {d.name}  ({d.stat().st_size / 1_000_000:.1f} MB)')